# Qwen3-TTS fine-tuning (VoiceStudio) - runner 23 (663eb0ab)

Trains Qwen3-TTS on the dataset folder VoiceStudio exported, using Qwen's official `finetuning/` scripts. Everything reusable is cached on your Google Drive, and every cell below calls one script (`voicestudio_qwen3_colab.py`, shipped with the export). **Nothing here has been run on Colab by its author: treat the first run as the test.**

**Version lock (the usual way this breaks).** FlashAttention is built for ONE exact Python + torch + CUDA combination. The last verified Colab runtime is Python 3.13 (cp313), torch 2.11.0+cu130, CUDA 13.0, transformers 4.57.3, with the community wheel `flash_attn-2.8.3+cu130torch2.11-cp313` (the official wheels have no CUDA 13 build). The setup cell finds the matching wheel itself and refuses one that does not fit. **Never upgrade or reinstall torch in the runtime**, and do not build FlashAttention by hand (about an hour, often out of memory). If Colab ships a newer torch or Python, the wheel will not match: the setup cell reports that in its pass/fail table before any training starts (SDPA is the slower fallback) instead of failing mid-run. Details: docs/engines/qwen3-tts-colab-runtime.md.

**Checklist**
1. **Runtime > Change runtime type > pick the GPU.** A100, H100 and L4 use the original script (FlashAttention 2). The **G4** (RTX PRO 6000 Blackwell) uses a patched SDPA copy by default (unverified). A **T4** cannot run FlashAttention 2 and needs `ATTN_FALLBACK = True` (unverified, may run out of memory).
2. **Compute units.** A runtime uses units while connected, idle or not, and each GPU type has its own rate (see Colab's runtime menu). Run once with `SMOKE_RUN = True` for a short timing run that projects time and units, then set it back to `False`.
3. Copy the export zip (or folder) to Drive and set `DATA_SOURCE` below to it.
4. Run the cells top to bottom. The first run on a new runtime type installs packages and downloads models; later runs reuse the Drive cache. FlashAttention is used only when a ready-made wheel exists for the runtime's exact Python, torch and CUDA: the runner looks one up in the community wheel releases (the pinned Python 3.13 / torch 2.11 / CUDA 13 wheel is the fallback). With no wheel, PyTorch's built-in attention (SDPA) is used, which gives the same training result (FlashAttention only changes speed and memory), and **nothing is compiled** unless you set `BUILD_FLASH_ATTN = True`.
5. When finished, **disconnect and delete the runtime** (last cell, or Runtime > Disconnect and delete runtime).
6. **To cancel fast:** press the stop button on the running cell (it stops the training and first saves the epochs that already finished), then **Runtime > Disconnect and delete runtime**. Compute units are used while the runtime is connected, not while a cell is running, so disconnecting is what stops the spending. Finished epochs stay on Drive; the epoch in progress is lost, because the official script cannot resume. If anything looks stuck, go straight to Disconnect and delete runtime.

In [ ]:
# GPU check: which GPU did Colab give you? (details and the attention path are printed again by the setup cell)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## Parameters
Edit the values in the next cell. `DATA_SOURCE` is where the training data is: the export zip or folder on Drive (or an already unzipped folder under `/content`). The cells after it do the work and hold no settings of their own.

In [ ]:
# ---- Edit these, then run the cells below in order. Re-run this cell and the 'mount Drive' cell after changing them. ----
DRIVE_ROOT = "/content/drive/MyDrive/Qwen3TTS"  # cache, checkpoints and logs live here
DATA_SOURCE = "/content/drive/MyDrive/Qwen3TTS/datasets/my-dataset.zip"  # where the training data is: a .zip or a folder on Drive, or an unzipped folder under /content
SPEAKER_NAME = "MySpeaker"  # the name the fine-tuned model answers to
EPOCHS = 10
LR = 2e-06  # learning rate of a FULL fine-tune (TRAIN_MODE = 'full')
TRAIN_MODE = "full"  # 'full': train the whole model (LR). 'lora': train small adapters on a frozen model (LORA_LR); every epoch is still saved as a normal merged checkpoint, plus the adapter. LoRA is UNVERIFIED on a GPU
LORA_LR = 2e-06  # learning rate of a LoRA run (owner-confirmed 2e-6)
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
INSTRUCT_CONDITIONING = False  # True: train with the instruction block that --instruct generation prepends (tags from the export become the instruction; UNVERIFIED on a GPU)
INSTRUCT_DROPOUT = 0.3  # chance a tagged clip is trained without its tags
LORA_SCALE = 0.3  # multiplier on alpha/r when each epoch's adapter is merged into its checkpoint. default 0.3 (range 0.0-1.0; warn above 0.5; 0.0 = base model)
SPEAKERS = ''  # several speakers in one model: comma-separated names. Empty: taken from the `speaker` of each clip in the export, else SPEAKER_NAME alone
FAST_SETUP = True  # True: clone the Qwen repo and download the models straight into /content and keep pip's cache there (saves minutes of slow Drive reads each session). Your data, codes, checkpoints and logs stay on Drive either way.
GRAD_ACCUM = 2  # micro-batches per update (the script fixes 4). BATCH_SIZE x GRAD_ACCUM = clips per update: 2 x 4 or 4 x 2 both make 8
BATCH_SIZE = 4  # a number, or 'auto' for the suggestion for this GPU (untested estimates, see the GPU section of the output)
BASE_MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
RUN_ID = ''  # empty: a new timestamped run folder every time, so a second run never overwrites the first
KEEP_ALL = True  # True keeps every checkpoint (each is several GB). Nothing is ever deleted while this is True
KEEP_LAST_N = 0  # only with KEEP_ALL = False: keep the newest N epochs (plus KEEP_FIRST / KEEP_FINAL)
KEEP_FIRST = False
KEEP_FINAL = True
ATTN_FALLBACK = 'auto'  # 'auto': FlashAttention 2 when a ready-made package exists for this runtime, otherwise PyTorch's built-in attention (patched copy of the script, unverified); True: always the patched copy; False: require FlashAttention 2
USE_VENV = 'auto'  # 'auto': when this runtime has no FlashAttention package, build an isolated Python 3.12 + torch environment that has one (a few minutes, once per runtime). True: always. False: never.
FLASH_WHEEL = "https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3%2Bcu130torch2.11-cp313-cp313-linux_x86_64.whl"  # a flash_attn .whl (Drive path or URL) for THIS runtime's Python/torch/CUDA, tried first; it is name-checked, never installed when it does not fit. Empty: the runner looks the right one up
BUILD_FLASH_ATTN = False  # True: compile FlashAttention from source when no ready-made package exists. Takes an hour or more, may fail on a new Python/CUDA, and uses compute units the whole time. Leave False.
RESTART = False  # True lets a RUN_ID you re-use replace its own checkpoints (the official script cannot resume)
SAMPLE_TEXT = "The lighthouse keeper counted seven ships before the fog rolled in."  # sentence for final.wav after training
SAMPLE_SEED = 1234
CACHE_MODELS_ON_DRIVE = False  # False: download models straight to /content (Drive only keeps your epochs). True also caches the models on Drive
VAL_FRACTION = 0.05  # share of your clips held out and scored every epoch (validation loss in metrics.csv). 0 turns it off
DRIVE_SKIP_FIRST = 3  # the first N epochs stay on /content and are not copied to Drive (the final epoch is always copied)
SMOKE_RUN = False  # True: a short timing run (one epoch on the first SMOKE_CLIPS clips, scratch folder) instead of real training
SMOKE_CLIPS = 40
UNITS_PER_HOUR = None  # this runtime's compute units per hour; empty = looked up for known GPUs (owner's figures, unverified)
UNITS_AVAILABLE = 100
COMPARE_EPOCH_SECONDS = None  # optional: seconds per epoch measured on a different GPU, to compare cost per epoch
COMPARE_UNITS_PER_HOUR = None  # that other GPU's rate
TOKENIZER_MODEL = "Qwen/Qwen3-TTS-Tokenizer-12Hz"

## 1. Mount Drive and load the script

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load the one script that does the work: from the export (DATA_SOURCE) if it is there, else the copy cached on Drive.
import os, sys, zipfile
SCRIPT = 'voicestudio_qwen3_colab.py'
if DRIVE_ROOT.startswith('/content/drive/'):
    # Never create the Drive folder from here: if Colab cannot see the real one yet, creating it makes a second folder with the same name.
    import time
    MY_DRIVE = '/content/drive/MyDrive'
    for _ in range(30):  # the mount can take a moment to fill in
        if os.path.isdir(MY_DRIVE) and os.listdir(MY_DRIVE):
            break
        time.sleep(2)
    if not os.path.isdir(DRIVE_ROOT):
        near = [n for n in sorted(os.listdir(MY_DRIVE)) if 'qwen' in n.lower() or 'tts' in n.lower()][:10] if os.path.isdir(MY_DRIVE) else []
        raise FileNotFoundError(DRIVE_ROOT + ' does not exist, so nothing was created. Create that folder in Google Drive with exactly this name and wait until it has finished syncing, or change DRIVE_ROOT. Similar folders in My Drive: ' + str(near or 'none'))
CACHE = os.path.join(DRIVE_ROOT, 'cache')
os.makedirs(CACHE, exist_ok=True)
cached = os.path.join(CACHE, SCRIPT)
import re
def _version(b):
    m = re.search(rb'^RUNNER_VERSION\s*=\s*"(\d+)"', b or b'', re.M)
    return int(m.group(1)) if m else 0
fresh = None
if DATA_SOURCE.lower().endswith('.zip') and os.path.isfile(DATA_SOURCE):
    with zipfile.ZipFile(DATA_SOURCE) as z:
        hits = [n for n in z.namelist() if n == SCRIPT or n.endswith('/' + SCRIPT)]
        fresh = z.read(hits[0]) if hits else None
elif os.path.isfile(os.path.join(DATA_SOURCE, SCRIPT)):
    fresh = open(os.path.join(DATA_SOURCE, SCRIPT), 'rb').read()
have = open(cached, 'rb').read() if os.path.isfile(cached) else None
import base64, zlib
# The runner script, stored inside this notebook (compressed), so nothing else has to be uploaded next to it.
EMBEDDED = zlib.decompress(base64.b64decode('...'))
# The newest runner wins (zip, this notebook, or the copy cached on Drive), so a fixed script never needs the zip to be rebuilt.
best = max([c for c in (fresh, EMBEDDED, have) if c is not None], key=_version, default=None)
if best is not None and best != have:
    open(cached, 'wb').write(best)
elif have is not None:
    print('Using the runner already cached in', cached)
if not os.path.isfile(cached):
    raise FileNotFoundError(SCRIPT + ' not found: check DATA_SOURCE (it must point at the export zip or folder, which contains it).')
if CACHE not in sys.path:
    sys.path.insert(0, CACHE)
import importlib, voicestudio_qwen3_colab as vs
importlib.reload(vs)
P = vs.params_from(dict(drive_root=DRIVE_ROOT, data_source=DATA_SOURCE, speaker_name=SPEAKER_NAME, epochs=EPOCHS, lr=LORA_LR if TRAIN_MODE == 'lora' else LR,
    train_mode=TRAIN_MODE, lora_rank=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, lora_scale=LORA_SCALE, instruct_conditioning=INSTRUCT_CONDITIONING, instruct_dropout=INSTRUCT_DROPOUT, speakers=SPEAKERS,
    batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM, fast_setup=FAST_SETUP, base_model=BASE_MODEL, tokenizer_model=TOKENIZER_MODEL, run_id=RUN_ID, keep_all=KEEP_ALL, keep_last_n=KEEP_LAST_N,
    keep_first=KEEP_FIRST, keep_final=KEEP_FINAL, sample_text=SAMPLE_TEXT, sample_seed=SAMPLE_SEED, cache_models_on_drive=CACHE_MODELS_ON_DRIVE, val_fraction=VAL_FRACTION, drive_skip_first=DRIVE_SKIP_FIRST,
    attn_fallback=ATTN_FALLBACK, use_venv=USE_VENV, build_flash_attn=BUILD_FLASH_ATTN, flash_wheel=FLASH_WHEEL, restart=RESTART, smoke_run=SMOKE_RUN, smoke_clips=SMOKE_CLIPS, units_per_hour=UNITS_PER_HOUR, units_available=UNITS_AVAILABLE,
    compare_epoch_seconds=COMPARE_EPOCH_SECONDS, compare_units_per_hour=COMPARE_UNITS_PER_HOUR), env={})
print('Runner', vs.RUNNER_VERSION, 'loaded from', cached)

## 2. Set up and verify
Creates the Drive layout, installs what is missing, gets FlashAttention (cached wheel, prebuilt wheel or one build saved to Drive) when the GPU needs it, downloads the models once, and prints a pass/fail table plus the time each step took.

In [ ]:
ok = vs.setup(P)

## 3. Check the data
Copies or unzips the dataset to local disk and verifies every `audio` and `ref_audio` path (exists, 24 kHz mono) before any GPU time is spent.

In [ ]:
folder = vs.data_check(P)

## 4. Train (or run the timing run)
With `SMOKE_RUN = True` this runs one epoch on the first few clips and prints the projected time and compute units; with `False` it trains for real. Checkpoints are written to `/content`, then each finished epoch is copied to `runs/<speaker>/<run_id>/checkpoints` on Drive under a temporary name, verified and renamed, and the output says what is now safe on Drive. Progress lines, a heartbeat when quiet, `metrics.csv` and `status.json` show it is alive. Re-running the cell starts a new run unless you re-use a `RUN_ID`.

In [ ]:
vs.train(P)

## 5. Results
Lists the checkpoints on Drive with their sizes.

In [ ]:
vs.results(P)

## 6. Final sample (optional, unverified)
Training already makes `samples/final.wav` when it finishes. Run this to make it again (for example with a new `SAMPLE_TEXT`) from the final checkpoint on Drive, using the usage in Qwen's finetuning README.

In [ ]:
vs.final_sample(P)

## 7. Disconnect
An idle runtime keeps using compute units. Uncomment and run the line below, or use Runtime > Disconnect and delete runtime. (`runtime.unassign()` was announced by the Colab team in 2022; it is not run here.)

In [ ]:
# from google.colab import runtime; runtime.unassign()